In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q transformers
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q pymupdf
!pip install -q accelerate
!pip install -q torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 51.0 MB/s eta 0:00:00


In [3]:
import torch

print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

GPU Available: False


In [4]:
import os

BASE_PATH = "/content/drive/MyDrive/DocuMind_RAG"

PDF_PATH = f"{BASE_PATH}/data/pdfs"
VECTOR_PATH = f"{BASE_PATH}/vector_db"
OUTPUT_PATH = f"{BASE_PATH}/outputs"

os.makedirs(VECTOR_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)

print("Folders Ready")

Folders Ready


In [5]:
import fitz

In [6]:
import fitz

def extract_pdf_text(pdf_path):

    doc = fitz.open(pdf_path)

    full_text = []
    metadata = []

    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text()

        full_text.append(text)

        metadata.append({
            "page": page_num + 1,
            "text": text
        })

    return "\n".join(full_text), metadata

In [7]:
print(extract_pdf_text)

<function extract_pdf_text at 0x7cf2d89eeca0>


In [8]:
pdf_file = f"{PDF_PATH}/AI_GPU_Summer_Internship_Program.pdf"

text, metadata = extract_pdf_text(pdf_file)

print(text[:2000])

 
 
 
 
 
 
 
 
SUMMER INTERNSHIP PROGRAM 
Artificial Intelligence & High-Performance Computing 
 
 
Powered by NVIDIA H200 GPU Infrastructure 
 
 
 
Program Duration 
2 Weeks (14 Days) 
Credit Hours 
2 Credits 
Total Contact Hours 
30 Hours 
Format 
Lecture + Hands-on GPU Lab 
Hardware 
NVIDIA H200 Tensor Core GPU 
Target Audience 
Undergraduate / Graduate Students 
Prerequisites 
Basic Python Programming 
 
 
Presidency School of AI and Advanced Computing, Presidency University 
Summer 2026 
 
 
Prepared 
Recommended & Verified 
Approved 
 
 
 
 
 
 
  
 
 
Dr. Robin Rohit Vincent 
Dr. Shakkeera L 
Dr. S. Sivaperumal 
 
Head AI CoE NVIDIA 
Associate Dean, PSCS(Spl) 
Pro Vice-Chancellor 

Presidency School of AI and Advanced Studies, Presidency University 
Summer Internship Program — 2026 
Page 2 
 
 
 
1. Program Overview 
 
This two-week intensive summer internship is designed to provide students with a rigorous, hands-on 
introduction to Artificial Intelligence, Machine Learning, a

In [9]:
import re

def clean_text(text):

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    # Remove extra newlines
    text = text.strip()

    return text

In [10]:
cleaned_text = clean_text(text)

print(cleaned_text[:1000])

SUMMER INTERNSHIP PROGRAM Artificial Intelligence & High-Performance Computing Powered by NVIDIA H200 GPU Infrastructure Program Duration 2 Weeks (14 Days) Credit Hours 2 Credits Total Contact Hours 30 Hours Format Lecture + Hands-on GPU Lab Hardware NVIDIA H200 Tensor Core GPU Target Audience Undergraduate / Graduate Students Prerequisites Basic Python Programming Presidency School of AI and Advanced Computing, Presidency University Summer 2026 Prepared Recommended & Verified Approved Dr. Robin Rohit Vincent Dr. Shakkeera L Dr. S. Sivaperumal Head AI CoE NVIDIA Associate Dean, PSCS(Spl) Pro Vice-Chancellor Presidency School of AI and Advanced Studies, Presidency University Summer Internship Program — 2026 Page 2 1. Program Overview This two-week intensive summer internship is designed to provide students with a rigorous, hands-on introduction to Artificial Intelligence, Machine Learning, and GPU-accelerated computing. Students will progress from foundational AI theory through applied 

In [11]:
def create_chunks(text, chunk_size=500, overlap=100):

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [12]:
chunks = create_chunks(cleaned_text)

print("Total Chunks:", len(chunks))

print("\nFirst Chunk:\n")
print(chunks[0])

Total Chunks: 39

First Chunk:

SUMMER INTERNSHIP PROGRAM Artificial Intelligence & High-Performance Computing Powered by NVIDIA H200 GPU Infrastructure Program Duration 2 Weeks (14 Days) Credit Hours 2 Credits Total Contact Hours 30 Hours Format Lecture + Hands-on GPU Lab Hardware NVIDIA H200 Tensor Core GPU Target Audience Undergraduate / Graduate Students Prerequisites Basic Python Programming Presidency School of AI and Advanced Computing, Presidency University Summer 2026 Prepared Recommended & Verified Approved Dr. Robin


In [13]:
for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i+1} ---\n")
    print(chunk)


--- Chunk 1 ---

SUMMER INTERNSHIP PROGRAM Artificial Intelligence & High-Performance Computing Powered by NVIDIA H200 GPU Infrastructure Program Duration 2 Weeks (14 Days) Credit Hours 2 Credits Total Contact Hours 30 Hours Format Lecture + Hands-on GPU Lab Hardware NVIDIA H200 Tensor Core GPU Target Audience Undergraduate / Graduate Students Prerequisites Basic Python Programming Presidency School of AI and Advanced Computing, Presidency University Summer 2026 Prepared Recommended & Verified Approved Dr. Robin

--- Chunk 2 ---

nced Computing, Presidency University Summer 2026 Prepared Recommended & Verified Approved Dr. Robin Rohit Vincent Dr. Shakkeera L Dr. S. Sivaperumal Head AI CoE NVIDIA Associate Dean, PSCS(Spl) Pro Vice-Chancellor Presidency School of AI and Advanced Studies, Presidency University Summer Internship Program — 2026 Page 2 1. Program Overview This two-week intensive summer internship is designed to provide students with a rigorous, hands-on introduction to Arti

In [14]:
!pip install -q sentence-transformers

In [15]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [16]:
import torch

print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU Available: False


In [17]:
embeddings = embedding_model.encode(
    chunks,
    convert_to_tensor=False,
    show_progress_bar=True
)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

In [18]:
print("Number of Chunks:", len(chunks))
print("Embedding Dimension:", len(embeddings[0]))

Number of Chunks: 39
Embedding Dimension: 384


In [19]:
import numpy as np

embeddings = np.array(embeddings).astype("float32")

print(embeddings.shape)

(39, 384)


In [20]:
np.save(
    "document_embeddings.npy",
    embeddings
)

print("Embeddings saved!")

Embeddings saved!


In [21]:
import faiss
import numpy as np

In [22]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("Total vectors stored:", index.ntotal)

Total vectors stored: 39


In [23]:
faiss.write_index(
    index,
    "documind_index.faiss"
)

print("FAISS index saved!")

FAISS index saved!


In [24]:
index = faiss.read_index(
    "documind_index.faiss"
)

print("Index loaded!")

Index loaded!


In [25]:
def retrieve_chunks(
    query,
    embedding_model,
    index,
    chunks,
    top_k=3
):

    query_embedding = embedding_model.encode(
        [query]
    )

    query_embedding = np.array(
        query_embedding
    ).astype("float32")

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for idx in indices[0]:
        results.append(chunks[idx])

    return results

In [26]:
query = "How long is the internship program?"

results = retrieve_chunks(
    query,
    embedding_model,
    index,
    chunks
)

for i, chunk in enumerate(results):
    print(f"\n--- Result {i+1} ---\n")
    print(chunk)


--- Result 1 ---

SUMMER INTERNSHIP PROGRAM Artificial Intelligence & High-Performance Computing Powered by NVIDIA H200 GPU Infrastructure Program Duration 2 Weeks (14 Days) Credit Hours 2 Credits Total Contact Hours 30 Hours Format Lecture + Hands-on GPU Lab Hardware NVIDIA H200 Tensor Core GPU Target Audience Undergraduate / Graduate Students Prerequisites Basic Python Programming Presidency School of AI and Advanced Computing, Presidency University Summer 2026 Prepared Recommended & Verified Approved Dr. Robin

--- Result 2 ---

is provided on the H200 cluster Presidency School of AI and Advanced Studies, Presidency University Summer Internship Program — 2026 Page 8 9. Program Benefits & Outcomes What Students Receive • 2 transferable academic credit hours upon successful completion • Completion certificate from the AI & GPU Computing Summer Internship Program Career Pathways • Machine Learning Engineer — model development, training pipeline optimization • AI Research Scientist — a

In [27]:
!pip install -q transformers sentencepiece accelerate

In [28]:
from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM

model_name = "google/flan-t5-large"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("FLAN-T5 loaded successfully!")

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

FLAN-T5 loaded successfully!


In [29]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

llm.to(device)

print("Running on:", device)

Running on: cpu


In [30]:
def generate_answer(context, question):

    prompt = f"""
You are a helpful assistant.

Read the context carefully.

Answer the question in complete sentences.

If the context contains a list, provide the full list.

Context:
{context}

Question:
{question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    outputs = llm.generate(
        **inputs,
        max_new_tokens=100
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [31]:
print("Model Device:", next(llm.parameters()).device)
print("Current Device:", device)

Model Device: cpu
Current Device: cpu


In [32]:
def ask_documind(question):

    retrieved_chunks = retrieve_chunks(
        question,
        embedding_model,
        index,
        chunks,
        top_k=5
    )

    context = "\n".join(retrieved_chunks)

    answer = generate_answer(
        context,
        question
    )

    return {
        "question": question,
        "answer": answer,
        "sources": retrieved_chunks
    }

In [33]:
result = ask_documind(
    "How long is the internship program?"
)

print("QUESTION:")
print(result["question"])

print("\nANSWER:")
print(result["answer"])

QUESTION:
How long is the internship program?

ANSWER:
2 Weeks (14 Days)


In [34]:
print("\nSOURCES:\n")

for i, source in enumerate(result["sources"]):
    print(f"\nSource {i+1}:")
    print(source[:300])


SOURCES:


Source 1:
SUMMER INTERNSHIP PROGRAM Artificial Intelligence & High-Performance Computing Powered by NVIDIA H200 GPU Infrastructure Program Duration 2 Weeks (14 Days) Credit Hours 2 Credits Total Contact Hours 30 Hours Format Lecture + Hands-on GPU Lab Hardware NVIDIA H200 Tensor Core GPU Target Audience Under

Source 2:
is provided on the H200 cluster Presidency School of AI and Advanced Studies, Presidency University Summer Internship Program — 2026 Page 8 9. Program Benefits & Outcomes What Students Receive • 2 transferable academic credit hours upon successful completion • Completion certificate from the AI & GP

Source 3:
 how hours are allocated across lectures, labs, and assessment: Activity Type Hours / Week Total Hours % of Program Lecture / Theory Sessions 3 hrs 6 hrs 20% Guided Coding Workshops 3 hrs 6 hrs 20% GPU Hands-on Lab Sessions 5 hrs 10 hrs 33% Mentored Project Work 3 hrs 6 hrs 20% Assessment & Presenta

Source 4:
OPS Interconnect NVLink 4 / NVSwitch Compu

In [35]:
{
    "chunk_id": 1,
    "page": 3,
    "text": "..."
}

{'chunk_id': 1, 'page': 3, 'text': '...'}

In [36]:
result = ask_documind(
    "How many credits are awarded?"
)

print(result)

{'question': 'How many credits are awarded?', 'answer': '2', 'sources': [' how hours are allocated across lectures, labs, and assessment: Activity Type Hours / Week Total Hours % of Program Lecture / Theory Sessions 3 hrs 6 hrs 20% Guided Coding Workshops 3 hrs 6 hrs 20% GPU Hands-on Lab Sessions 5 hrs 10 hrs 33% Mentored Project Work 3 hrs 6 hrs 20% Assessment & Presentations 1 hr 2 hrs 7% TOTAL 15 hrs/week 30 hrs 100% Credit Equivalency: 30 contact hours = 2 academic credit hours (following standard 15:1 hour-to-credit ratio) 3. Detailed 2-Week Schedule Week 1 — AI ', 's (7 labs × 5 pts) 35% End of each lab day Week 1 Quiz (AI Fundamentals) 10% Day 5 Capstone Project — Code & Repository 20% Day 9 (11:59 PM) Capstone Technical Report 15% Day 10 (Morning) Capstone Presentation & Demo 15% Day 10 (Afternoon) Peer Review & Participation 5% Day 10 TOTAL 100% 8. Program Policies & Requirements 8.1 Attendance • Attendance at all 10 instructional days is mandatory for credit • Students missin

In [37]:
!pip install -q gradio

In [38]:
current_chunks = None
current_index = None

def process_pdf(pdf_file):

    try:

        global current_chunks
        global current_index

        # Gradio uploads a file object
        pdf_path = pdf_file.name

        text, metadata = extract_pdf_text(pdf_path)

        cleaned_text = clean_text(text)

        current_chunks = create_chunks(
            cleaned_text,
            chunk_size=500,
            overlap=100
        )

        embeddings = generate_embeddings(
            current_chunks
        )

        current_index = create_index(
            embeddings
        )

        return f"""
PDF processed successfully!

Chunks Created: {len(current_chunks)}

Vector Database Ready.
"""

    except Exception as e:

        return f"ERROR:\n{str(e)}"

In [39]:
def answer_question(question):

    global current_chunks
    global current_index

    if current_index is None:

        return "Please upload a PDF first."

    retrieved_chunks = retrieve_chunks(
        question,
        embedding_model,
        current_index,
        current_chunks,
        top_k=3
    )

    context = "\n".join(
        retrieved_chunks
    )

    answer = generate_answer(
        context,
        question
    )

    sources = "\n\n".join(
        [
            f"Source {i+1}:\n{chunk[:200]}"
            for i, chunk in enumerate(
                retrieved_chunks
            )
        ]
    )

    return f"""
Answer:

{answer}

---------------------------------

Sources:

{sources}
"""

In [40]:
import gradio as gr

def process_pdf_gradio(pdf_file):

    global chunk_data
    global index

    try:

        text, metadata = extract_pdf_text(
            pdf_file.name
        )

        chunk_data = create_chunks_with_pages(
            metadata
        )

        chunks = [
            item["text"]
            for item in chunk_data
        ]

        embeddings = generate_embeddings(
            chunks
        )

        index = create_index(
            embeddings
        )

        return f"""
PDF Processed Successfully

Pages: {len(metadata)}
Chunks: {len(chunk_data)}

Vector Database Ready
"""

    except Exception as e:

        return f"Error: {str(e)}"


def ask_question(question):

    try:

        result = ask_documind(question)

        pages = ", ".join(
            [
                f"Page {p}"
                for p in result["pages"]
            ]
        )

        return (
            result["answer"],
            pages
        )

    except Exception as e:

        return (
            f"Error: {str(e)}",
            ""
        )


with gr.Blocks() as demo:

    gr.Markdown(
        "# DocuMind RAG"
    )

    gr.Markdown(
        "Intelligent Document Question Answering System"
    )

    pdf_file = gr.File(
        label="Upload PDF"
    )

    process_btn = gr.Button(
        "Process PDF"
    )

    status_box = gr.Textbox(
        label="Processing Status"
    )

    process_btn.click(
        process_pdf_gradio,
        inputs=pdf_file,
        outputs=status_box
    )

    gr.Markdown("---")

    question_box = gr.Textbox(
        label="Ask a Question"
    )

    ask_btn = gr.Button(
        "Get Answer"
    )

    answer_box = gr.Textbox(
        label="Answer",
        lines=6
    )

    citation_box = gr.Textbox(
        label="Source Pages"
    )

    ask_btn.click(
        ask_question,
        inputs=question_box,
        outputs=[
            answer_box,
            citation_box
        ]
    )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://efd061837a431d0749.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [41]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

def generate_embeddings(chunks):

    embeddings = embedding_model.encode(
        chunks,
        convert_to_tensor=False,
        show_progress_bar=True
    )

    return embeddings

print("Embedding module loaded!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding module loaded!


In [42]:
print(type(generate_embeddings))

<class 'function'>


In [79]:
print(type(create_index))

<class 'function'>


In [54]:
import faiss
import numpy as np

def create_index(embeddings):

    embeddings = np.array(
        embeddings
    ).astype("float32")

    dimension = embeddings.shape[1]

    index = faiss.IndexFlatL2(
        dimension
    )

    index.add(embeddings)

    return index

print("FAISS module loaded!")

FAISS module loaded!


In [55]:
import os

for root, dirs, files in os.walk("."):
    for file in files:
        if file.endswith(".pdf"):
            print(os.path.join(root, file))

./drive/MyDrive/DocuMind_RAG/data/pdfs/AI_GPU_Summer_Internship_Program.pdf


In [56]:
pdf_file = "/content/drive/MyDrive/DocuMind_RAG/data/pdfs/AI_GPU_Summer_Internship_Program.pdf"

text, metadata = extract_pdf_text(pdf_file)

print(text[:500])

 
 
 
 
 
 
 
 
SUMMER INTERNSHIP PROGRAM 
Artificial Intelligence & High-Performance Computing 
 
 
Powered by NVIDIA H200 GPU Infrastructure 
 
 
 
Program Duration 
2 Weeks (14 Days) 
Credit Hours 
2 Credits 
Total Contact Hours 
30 Hours 
Format 
Lecture + Hands-on GPU Lab 
Hardware 
NVIDIA H200 Tensor Core GPU 
Target Audience 
Undergraduate / Graduate Students 
Prerequisites 
Basic Python Programming 
 
 
Presidency School of AI and Advanced Computing, Presidency University 
Summer 2026 
 


In [57]:
import os
print(os.getcwd())

/content


In [58]:
import os

for root, dirs, files in os.walk("/content"):
    for file in files:
        if file.endswith(".pdf"):
            print(os.path.join(root, file))

/content/drive/MyDrive/DocuMind_RAG/data/pdfs/AI_GPU_Summer_Internship_Program.pdf


In [59]:
pdf_file = "/content/drive/MyDrive/DocuMind_RAG/data/pdfs/AI_GPU_Summer_Internship_Program.pdf"

text, metadata = extract_pdf_text(pdf_file)

cleaned_text = clean_text(text)

chunks = create_chunks(cleaned_text)

embeddings = generate_embeddings(chunks)

index = create_index(embeddings)

print("SUCCESS")
print("Chunks:", len(chunks))
print("Embeddings Shape:", embeddings.shape)
print("FAISS Vectors:", index.ntotal)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

SUCCESS
Chunks: 39
Embeddings Shape: (39, 384)
FAISS Vectors: 39


In [60]:
import numpy as np

def retrieve_chunks(
    query,
    embedding_model,
    index,
    chunks,
    top_k=3
):

    query_embedding = embedding_model.encode([query])

    query_embedding = np.array(
        query_embedding
    ).astype("float32")

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for idx in indices[0]:
        results.append(chunks[idx])

    return results

In [61]:
results = retrieve_chunks(
    "How many credits are awarded?",
    embedding_model,
    index,
    chunks
)

for i, chunk in enumerate(results):
    print(f"\n----- Result {i+1} -----\n")
    print(chunk)


----- Result 1 -----

 how hours are allocated across lectures, labs, and assessment: Activity Type Hours / Week Total Hours % of Program Lecture / Theory Sessions 3 hrs 6 hrs 20% Guided Coding Workshops 3 hrs 6 hrs 20% GPU Hands-on Lab Sessions 5 hrs 10 hrs 33% Mentored Project Work 3 hrs 6 hrs 20% Assessment & Presentations 1 hr 2 hrs 7% TOTAL 15 hrs/week 30 hrs 100% Credit Equivalency: 30 contact hours = 2 academic credit hours (following standard 15:1 hour-to-credit ratio) 3. Detailed 2-Week Schedule Week 1 — AI 

----- Result 2 -----

s (7 labs × 5 pts) 35% End of each lab day Week 1 Quiz (AI Fundamentals) 10% Day 5 Capstone Project — Code & Repository 20% Day 9 (11:59 PM) Capstone Technical Report 15% Day 10 (Morning) Capstone Presentation & Demo 15% Day 10 (Afternoon) Peer Review & Participation 5% Day 10 TOTAL 100% 8. Program Policies & Requirements 8.1 Attendance • Attendance at all 10 instructional days is mandatory for credit • Students missing more than 1 day without prior

In [62]:
from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("LLM Loaded Successfully")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

LLM Loaded Successfully


In [63]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

llm.to(device)

def generate_answer(context, question):

    prompt = f"""
Answer the question using only the provided context.

Context:
{context}

Question:
{question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    outputs = llm.generate(
        **inputs,
        max_new_tokens=100
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

In [64]:
question = "How many credits are awarded?"

retrieved = retrieve_chunks(
    question,
    embedding_model,
    index,
    chunks
)

context = "\n".join(retrieved)

answer = generate_answer(
    context,
    question
)

print("QUESTION:")
print(question)

print("\nANSWER:")
print(answer)

QUESTION:
How many credits are awarded?

ANSWER:
TOTAL 15 hrs/week 30 hrs 100% Credit Equivalency: 30 contact hours = 2 academic credit hours (following standard 15:1 hour-to-credit ratio) 3. Detailed 2-Week Schedule Week 1 — AI s (7 labs  5 pts) 35% End of each lab day Week 1 Quiz (AI Fundamentals) 10% Day 5 Capstone Project — Code & Repository 20% Day 9 (11:59 PM


In [65]:
question = "How many credits are awarded?"

retrieved = retrieve_chunks(
    question,
    embedding_model,
    index,
    chunks
)

context = "\n".join(retrieved)

answer = generate_answer(
    context,
    question
)

print(answer)

TOTAL 15 hrs/week 30 hrs 100% Credit Equivalency: 30 contact hours = 2 academic credit hours (following standard 15:1 hour-to-credit ratio) 3. Detailed 2-Week Schedule Week 1 — AI s (7 labs  5 pts) 35% End of each lab day Week 1 Quiz (AI Fundamentals) 10% Day 5 Capstone Project — Code & Repository 20% Day 9 (11:59 PM


In [66]:
def ask_documind(question):

    retrieved_chunks = retrieve_chunks(
        question,
        embedding_model,
        index,
        chunks,
        top_k=5
    )

    context = "\n".join(retrieved_chunks)

    answer = generate_answer(
        context,
        question
    )

    return {
        "question": question,
        "answer": answer,
        "sources": retrieved_chunks
    }

In [67]:
result = ask_documind(
    "What is the duration of the internship?"
)

print(result["answer"])

2 Weeks (14 Days)


In [68]:
def create_chunks_with_pages(
    metadata,
    chunk_size=500,
    overlap=100
):

    chunk_data = []

    for page_info in metadata:

        page_num = page_info["page"]
        text = page_info["text"]

        start = 0

        while start < len(text):

            end = start + chunk_size

            chunk = text[start:end]

            chunk_data.append({
                "page": page_num,
                "text": chunk
            })

            start += chunk_size - overlap

    return chunk_data

In [69]:
chunks = create_chunks(cleaned_text)

In [70]:
chunk_data = create_chunks_with_pages(metadata)

chunks = [
    item["text"]
    for item in chunk_data
]

In [71]:
def retrieve_chunks(
    query,
    embedding_model,
    index,
    chunk_data,
    top_k=3
):

    query_embedding = embedding_model.encode(
        [query]
    )

    query_embedding = np.array(
        query_embedding
    ).astype("float32")

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for idx in indices[0]:

        results.append(
            chunk_data[idx]
        )

    return results

In [72]:
retrieved = retrieve_chunks(
    question,
    embedding_model,
    index,
    chunk_data
)

context = "\n".join(
    [
        item["text"]
        for item in retrieved
    ]
)

In [73]:
print("\nSources:")

for item in retrieved:

    print(
        f"Page {item['page']}"
    )


Sources:
Page 3
Page 6
Page 3


In [74]:
question = "What is the duration of the program?"

retrieved = retrieve_chunks(
    question,
    embedding_model,
    index,
    chunk_data
)

for item in retrieved:
    print("Page:", item["page"])

Page: 3
Page: 6
Page: 1


In [75]:
def ask_documind(question):

    retrieved = retrieve_chunks(
        question,
        embedding_model,
        index,
        chunk_data,
        top_k=5
    )

    context = "\n".join(
        [item["text"] for item in retrieved]
    )

    answer = generate_answer(
        context,
        question
    )

    pages = sorted(
        list(
            set(
                item["page"]
                for item in retrieved
            )
        )
    )

    return {
        "answer": answer,
        "pages": pages
    }

In [76]:
result = ask_documind(
    "How many credits are awarded?"
)

print("Answer:")
print(result["answer"])

print("\nPages:")
print(result["pages"])

Answer:
2

Pages:
[1, 3, 6, 7]


In [77]:
result = ask_documind(
    "How many credits are awarded?"
)

print(result["answer"])
print(result["pages"])

2
[1, 3, 6, 7]


In [50]:
import faiss

faiss.write_index(
    index,
    "/content/drive/MyDrive/DocuMind_RAG/vector_db/documind_index.faiss"
)

print("Index Saved")

Index Saved


In [48]:
index = faiss.read_index(
    "/content/drive/MyDrive/DocuMind_RAG/vector_db/documind_index.faiss"
)

In [78]:
question = "is there any project choices and give the details"

retrieved = retrieve_chunks(
    question,
    embedding_model,
    index,
    chunk_data,
    top_k=3
)

for i, item in enumerate(retrieved):
    print(f"\n===== PAGE {item['page']} =====\n")
    print(item["text"])


===== PAGE 5 =====

LoRA 
LLaMA 3 
3 hrs 
Fine-tune LLaMA 3 8B with 4-bit 
QLoRA on custom data 
5 
Stable Diffusion Image 
Generation 
Diffusion Model 
2 hrs 
Generate & customize images with 
SDXL pipeline 


===== PAGE 6 =====

rHub on dedicated H200 GPU cluster — 1 GPU per student workload 
• 
CUDA Toolkit 12.x, cuDNN 9.x, cuBLAS, cuSPARSE 
• 
PyTorch 2.x with CUDA 12 support, torchvision, torchaudio 
• 
Hugging Face Transformers, Datasets, PEFT, TRL, Accelerate 
• 
NVIDIA Nsight Systems, Nsight Compute for GPU profiling 
• 
vLLM for high-throughput LLM inference serving 
• 
Weights & Biases (W&B) for experiment tracking and visualization 
• 
Git / GitHub for version control and lab submission 
 
6. Capstone Project 


===== PAGE 6 =====

mputer Vision 
• 
Object detection with YOLO v9 on a custom dataset 
• 
Medical image segmentation using U-Net or SAM (Segment Anything Model) 
• 
Video understanding with TimeSformer or VideoMAE 
Track B — Natural Language Processing 
• 
Domain-

In [46]:
print("Model Device:", next(llm.parameters()).device)
print("Current Device:", device)

Model Device: cpu
Current Device: cpu


In [45]:
device = "cpu"

llm = llm.to(device)

print("Model moved to CPU")

Model moved to CPU


In [80]:
device = "cpu"
llm = llm.to(device)

In [81]:
import gradio as gr

# Process PDF
def process_pdf_gradio(pdf_file):

    global chunk_data
    global index

    try:

        text, metadata = extract_pdf_text(
            pdf_file.name
        )

        chunk_data = create_chunks_with_pages(
            metadata
        )

        chunks = [
            item["text"]
            for item in chunk_data
        ]

        embeddings = generate_embeddings(
            chunks
        )

        index = create_index(
            embeddings
        )

        return f"""
PDF Processed Successfully

Pages: {len(metadata)}
Chunks: {len(chunk_data)}

Vector Database Ready
"""

    except Exception as e:

        return f"Error: {str(e)}"


# Question Answering
def ask_question(question):

    try:

        result = ask_documind(question)

        pages = ", ".join(
            [
                f"Page {p}"
                for p in result["pages"]
            ]
        )

        return (
            result["answer"],
            pages
        )

    except Exception as e:

        return (
            f"Error: {str(e)}",
            ""
        )


# Gradio UI
with gr.Blocks(title="DocuMind RAG") as demo:

    gr.Markdown(
        """
# 📚 DocuMind RAG

### Intelligent Document Question Answering System

Upload a PDF document and ask questions about its content.
"""
    )

    with gr.Row():

        pdf_file = gr.File(
            label="Upload PDF",
            file_types=[".pdf"]
        )

    process_btn = gr.Button(
        "Process PDF"
    )

    status_box = gr.Textbox(
        label="Processing Status"
    )

    process_btn.click(
        process_pdf_gradio,
        inputs=pdf_file,
        outputs=status_box
    )

    gr.Markdown("---")

    question_box = gr.Textbox(
        label="Ask a Question",
        placeholder="Example: What is the duration of the internship?"
    )

    ask_btn = gr.Button(
        "Get Answer"
    )

    answer_box = gr.Textbox(
        label="Answer",
        lines=6
    )

    citation_box = gr.Textbox(
        label="Source Pages"
    )

    ask_btn.click(
        ask_question,
        inputs=question_box,
        outputs=[
            answer_box,
            citation_box
        ]
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://80ddb56a489251b149.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [82]:
%%writefile requirements.txt
torch
transformers
sentence-transformers
faiss-cpu
pymupdf
gradio
numpy
accelerate

Writing requirements.txt


In [83]:
from google.colab import files

files.download("requirements.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>